In [31]:
import os
import random
import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torch.utils.data import random_split

from torchvision import transforms

from sklearn.metrics import accuracy_score

from tqdm import tqdm

In [32]:
DATA_DIR = "data/Processed_Train/images"
CSV_PATH = "data/Processed_Train/processed_annotations.csv"

MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

In [33]:
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 1e-3 # 0.001
TRAIN_RATIO = 0.8
RANDOM_SEED = 42

In [34]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(DEVICE)

cuda


In [35]:
train_transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomRotation(15),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),

    transforms.ToTensor(),

])

In [36]:
valid_transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.ToTensor(),

])

In [37]:
from sklearn.model_selection import train_test_split

annotations = pd.read_csv(CSV_PATH)

train_df, valid_df = train_test_split(
    annotations,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=annotations["Label"]
)

In [38]:
class SnakeDataset(Dataset):

    def __init__(self, annotations, image_dir, transform=None):

        self.annotations = annotations.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, index):

        row = self.annotations.iloc[index]

        image_path = os.path.join(
            self.image_dir,
            row["filename"]
        )

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = int(row["Label"])

        return image, label

In [39]:
train_dataset = SnakeDataset(
    annotations=train_df,
    image_dir=DATA_DIR,
    transform=train_transform
)

valid_dataset = SnakeDataset(
    annotations=valid_df,
    image_dir=DATA_DIR,
    transform=valid_transform
)

In [40]:
print("Total Images:", len(train_dataset))

Total Images: 4887


In [41]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    pin_memory=torch.cuda.is_available()
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=32,
    shuffle=False,
    pin_memory=torch.cuda.is_available()
)

In [42]:
print(len(annotations))
print(len(train_df))
print(len(valid_df))

print(train_df.head())

6109
4887
1222
                                               filename  \
4740  188473055_jpeg.rf.466d7b0f63343cfd34824576ed89...   
409   80818505_jpg.rf.177954c7705d0d0096e7f0d7c628c0...   
2368  127392976_jpeg.rf.7f45516d27de13bbdf1129186c44...   
1482  110829487_jpeg.rf.7e62aa70db88a8a79fb0e7a1ad06...   
1381  38830437_jpeg.rf.42361e17f36bac398f425a92ec2e2...   

                        class  Label  width  height  
4740       ophiophagus hannah      8    237     357  
409        chrysopelea ornata      4    212     500  
2368  trimeresurus albolabris     12    145     204  
1482      dendrelaphis pictus      6    375     402  
1381          daboia russelii      5    421     375  


In [44]:
print(len(train_dataset))
print(len(valid_dataset))

4887
1222


In [45]:
from torchvision.models import efficientnet_b0
from torchvision.models import EfficientNet_B0_Weights

weights = EfficientNet_B0_Weights.DEFAULT

model = efficientnet_b0(weights=weights)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\kisha/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:02<00:00, 8.64MB/s]


In [47]:
NUM_CLASSES = 15

model.classifier[1] = nn.Linear(
    in_features=model.classifier[1].in_features,
    out_features=NUM_CLASSES
)

In [48]:
model = model.to(DEVICE)

In [49]:
print(model.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=15, bias=True)
)


In [50]:
criterion = nn.CrossEntropyLoss()

In [51]:
optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4
)

In [52]:
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

In [53]:
scaler = torch.amp.GradScaler("cuda")

In [54]:
best_accuracy = 0.0
patience = 5
counter = 0

In [55]:
MODEL_PATH = "models/best_model.pth"

In [56]:
def train_one_epoch(model, dataloader, criterion, optimizer, scaler, device):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader, desc="Training", leave=False):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast(device_type="cuda"):

            outputs = model(images)

            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(dataloader)

    epoch_accuracy = 100 * correct / total

    return epoch_loss, epoch_accuracy

In [57]:
train_loss, train_acc = train_one_epoch(
    model,
    train_loader,
    criterion,
    optimizer,
    scaler,
    DEVICE
)

print(train_loss)
print(train_acc)

1.5458253867485945
51.83138940045017


In [59]:
def validate(model, dataloader, criterion, device):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in tqdm(dataloader, desc="Validation", leave=False):

            images = images.to(device)
            labels = labels.to(device)

            with torch.amp.autocast(device_type="cuda"):

                outputs = model(images)

                loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(dataloader)
    epoch_accuracy = 100 * correct / total

    return epoch_loss, epoch_accuracy

In [60]:
val_loss, val_acc = validate(
    model,
    valid_loader,
    criterion,
    DEVICE
)

print(val_loss)
print(val_acc)

1.0416172192646906
66.12111292962356


In [61]:
history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

In [63]:
def train_model():

    global best_accuracy
    global counter

    for epoch in range(EPOCHS):

        print(f"\nEpoch {epoch+1}/{EPOCHS}")
        print("-"*50)

        train_loss, train_acc = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            scaler,
            DEVICE
        )

        val_loss, val_acc = validate(
            model,
            valid_loader,
            criterion,
            DEVICE
        )

        scheduler.step(val_acc)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(f"Train Loss : {train_loss:.4f}")
        print(f"Train Acc  : {train_acc:.2f}%")

        print(f"Val Loss   : {val_loss:.4f}")
        print(f"Val Acc    : {val_acc:.2f}%")

        if val_acc > best_accuracy:

            best_accuracy = val_acc

            counter = 0

            checkpoint = {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_accuracy": best_accuracy
            }

            torch.save(checkpoint, MODEL_PATH)
            
            print("✅ Best model saved")

        else:

            counter += 1

            print(f"No improvement ({counter}/{patience})")

        if counter >= patience:

            print("\nEarly stopping triggered")

            break

In [ ]:
train_model()


Epoch 1/20
--------------------------------------------------


Train Loss : 0.8334
Train Acc  : 73.13%
Val Loss   : 0.6835
Val Acc    : 78.97%
✅ Best model saved

Epoch 2/20
--------------------------------------------------


Train Loss : 0.5606
Train Acc  : 82.30%
Val Loss   : 0.5565
Val Acc    : 84.12%
✅ Best model saved

Epoch 3/20
--------------------------------------------------


Train Loss : 0.4396
Train Acc  : 86.23%
Val Loss   : 0.4746
Val Acc    : 84.86%
✅ Best model saved

Epoch 4/20
--------------------------------------------------


Train Loss : 0.3116
Train Acc  : 89.69%
Val Loss   : 0.4688
Val Acc    : 86.09%
✅ Best model saved

Epoch 5/20
--------------------------------------------------


...

Train Loss : 0.1843
Train Acc  : 94.00%
Val Loss   : 0.4059
Val Acc    : 89.36%
No improvement (2/5)

Epoch 10/20
--------------------------------------------------


Train Loss : 0.1560
Train Acc  : 94.90%
Val Loss   : 0.3470
Val Acc    : 90.26%
No improvement (3/5)

Epoch 11/20
--------------------------------------------------


Train Loss : 0.0758
Train Acc  : 97.59%
Val Loss   : 0.1199
Val Acc    : 96.07%
✅ Best model saved

Epoch 12/20
--------------------------------------------------


Train Loss : 0.0397
Train Acc  : 98.81%
Val Loss   : 0.1381
Val Acc    : 95.91%
No improvement (1/5)

Epoch 13/20
--------------------------------------------------


Training:  34%|███▍      | 52/153 [00:36<01:08,  1.47it/s]